# Granite Speech Demo — full stack in Colab

This notebook spins up the entire [granite-speech-demo](https://github.com/generative-computing/mellea-demos/tree/main/2026-granite-speech) stack inside one Colab runtime — both vLLM model servers (Granite Speech 4.1 STT + Granite Switch 4.1 LLM), the Pipecat backend, and the Next.js frontend — then prints a public URL you open in your browser to start talking.

**Browser mic → WebRTC → Granite Speech STT → Mellea/Granite Switch LLM → Kokoro TTS → browser speaker.**

## Prerequisites

- **GPU runtime: A100 (Colab Pro) recommended.** L4 works. T4 will OOM — both Granite models won't fit.
- **HuggingFace read token.** Free; create one at https://huggingface.co/settings/tokens. Add it as a Colab Secret named `HF_TOKEN` (sidebar → 🔑 → New secret). Used for two things: downloading the Granite model weights, *and* minting per-session WebRTC TURN credentials so audio reaches your browser.
- **Browser:** Chrome, Edge, or Firefox. Safari may behave oddly with WebRTC.

## How long this takes

- **First run on a fresh runtime: ~8–10 min** (model downloads dominate).
- **Subsequent runs with weights cached: ~3 min.**

## What to do

1. Set the `HF_TOKEN` Colab Secret.
2. Switch the runtime to a GPU (Runtime → Change runtime type → A100/L4).
3. **Runtime → Run all.**
4. When the last cell prints a `*.trycloudflare.com` URL, open it, allow mic access, and start talking.

If anything goes wrong, scroll to the bottom — there's a troubleshooting section and a kill-switch cell.

## Cell 2 — Install dependencies (~3 min)

Clones the repo, installs Python deps via `uv`, installs frontend deps via `npm`, and downloads the `cloudflared` binary used for the public tunnel.

In [ ]:
import subprocess, os, shutil

def sh(cmd, **kwargs):
    print(f"\n$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, **kwargs)

# Re-runnable: nuke any stale clones so we don't trip on existing dirs.
shutil.rmtree("mellea-demos", ignore_errors=True)
shutil.rmtree("/tmp/granite-switch", ignore_errors=True)

# Colab's default Ubuntu repo has Node 12, which is too old for Next.js
# (chokes on optional-chaining). Install Node 20 from NodeSource instead.
sh("curl -fsSL https://deb.nodesource.com/setup_20.x | bash -")
sh("apt-get -qq install -y nodejs")

sh("git clone -b notebook https://github.com/psschwei/mellea-demos")
os.chdir("mellea-demos/2026-granite-speech")
print("cwd:", os.getcwd())

sh("pip install -q uv")
sh("uv sync")

# IMPORTANT: uv sync creates .venv, but `uv pip install` by default targets
# the system Python. Pin every subsequent install to the project venv.
VENV_PY = os.path.abspath(".venv/bin/python")
assert os.path.exists(VENV_PY), f"venv missing: {VENV_PY}"

# The install order below is load-bearing. Each step's pins can override
# the previous step's resolution; the final order leaves us with:
#   - mellea @ main (provides register_embedded_adapter_model, missing in 0.4.2)
#   - vllm 0.19.x with [audio] extras (Granite Speech needs librosa)
#   - granite_switch model architecture registered
#   - transformers 5.5.1 (older versions truncate the requirement_check JSON;
#     newer versions might or might not, so pin exactly what we tested)

# 1. mellea from main (0.4.2 release lacks APIs the demo uses)
sh(f"uv pip install --python {VENV_PY} 'mellea[all] @ git+https://github.com/generative-computing/mellea@main'")

# 2. vllm + the right transformers floor + granite_switch model registration.
#    The granite-switch repo's [vllm] extra pins vllm >=0.19.1,<0.21.0 and
#    transformers >=5.5.1 — installing plain `pip install vllm` gives 0.21.0
#    with an older transformers, which fails to recognize the architecture.
sh("git clone https://github.com/generative-computing/granite-switch /tmp/granite-switch")
assert os.path.exists("/tmp/granite-switch/pyproject.toml"), "granite-switch clone failed"
sh(f"uv pip install --python {VENV_PY} -e '/tmp/granite-switch[vllm]'")

# 3. vllm audio extras — without this, /v1/chat/completions returns 500 with
#    'Please install vllm[audio] for audio support' on any audio input.
sh(f"uv pip install --python {VENV_PY} 'vllm[audio]'")

# 4. Final transformers pin. The earlier installs can leave us on 4.57.6
#    (GPT2 tokenizer crashes on Granite Switch) or 5.0.0 (works for chat
#    but truncates requirement_check JSON output). 5.5.1 is what we tested.
sh(f"uv pip install --python {VENV_PY} 'transformers==5.5.1'")

sh("cd frontend && npm install --silent")
sh("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared")
sh("chmod +x /usr/local/bin/cloudflared")

# Sanity checks — explicitly use the venv's Python so we're checking the right env.
sh(f"{VENV_PY} -c 'import vllm; print(\"vllm version:\", vllm.__version__)'")
sh(f"{VENV_PY} -c 'import granite_switch.hf'")
sh(f"{VENV_PY} -c 'import transformers; v = transformers.__version__; assert v == \"5.5.1\", f\"got {v}, wanted 5.5.1\"; print(\"transformers OK:\", v)'")
sh(f"{VENV_PY} -c 'from mellea.backends.openai import OpenAIBackend; assert hasattr(OpenAIBackend, \"register_embedded_adapter_model\"), \"mellea version too old\"; print(\"mellea OK\")'")
sh(f"{VENV_PY} -c 'import librosa; print(\"vllm audio extras OK (librosa\", librosa.__version__, \")\")'")

print("\n✅ Install complete")

## Cell 3 — Configure secrets (instant)

Reads `HF_TOKEN` from Colab Secrets and exports it. Used for both HuggingFace model downloads and per-session TURN credential minting (see [TURN setup](https://turn.fastrtc.org/) — Cloudflare-backed, 10GB/mo free per HF token).

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("✅ HF_TOKEN configured — TURN credentials will be minted per-session")

## Cell 4 — Launch vLLM model servers (~5–8 min cold, ~30s cached)

Two vLLM processes:
- **Port 8083:** [`ibm-granite/granite-speech-4.1-2b`](https://huggingface.co/ibm-granite/granite-speech-4.1-2b) — STT.
- **Port 8000:** [`ibm-granite/granite-switch-4.1-3b-preview`](https://huggingface.co/ibm-granite/granite-switch-4.1-3b-preview) — chat LLM with `requirement_check` ALoRA intrinsics.

Both run in the background; logs stream to `logs/vllm-*.log`. The cell blocks until both servers respond on `/v1/models`.

In [ ]:
import os
import subprocess
import time
import urllib.request
import urllib.error

os.makedirs("logs", exist_ok=True)

VENV_VLLM = os.path.abspath(".venv/bin/vllm")
assert os.path.exists(VENV_VLLM), f"vllm not installed in venv: {VENV_VLLM}"

# Pre-flight: kill any stale vllm processes from a prior failed run, then
# verify the GPU has enough free memory before we try again.
subprocess.run("pkill -9 -f vllm || true", shell=True)
time.sleep(3)
free_mem = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"]
).decode().strip().splitlines()[0]
free_gib = int(free_mem) / 1024
print(f"GPU free memory: {free_gib:.1f} GiB")
if free_gib < 22:
    raise RuntimeError(
        f"Only {free_gib:.1f} GiB free on the GPU — need >=22. Something else is using it.\n"
        "Run `!nvidia-smi` in a new cell to see which process. Kill it with `!kill -9 <PID>`."
    )

def _tail(path: str, n: int = 80) -> str:
    try:
        with open(path) as f:
            return "".join(f.readlines()[-n:])
    except FileNotFoundError:
        return "(log file missing)"

def wait_for(url: str, name: str, proc: subprocess.Popen, log_path: str, timeout: int = 1200) -> None:
    """Poll until the URL returns 2xx. Bails out early if the process dies."""
    start = time.time()
    last_err = None
    while time.time() - start < timeout:
        rc = proc.poll()
        if rc is not None:
            raise RuntimeError(
                f"{name} exited early with code {rc}. Last log lines:\n"
                + "-" * 60 + "\n" + _tail(log_path) + "-" * 60
            )
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if 200 <= r.status < 300:
                    elapsed = int(time.time() - start)
                    print(f"✅ {name} ready ({elapsed}s)")
                    return
        except urllib.error.HTTPError as e:
            # vllm returns 401 to unauth'd /v1/models polls when --api-key is set.
            # The 401 proves the server is up and accepting requests, which is
            # all we care about for readiness. Any HTTPError means the server
            # is responding, so treat it as ready.
            elapsed = int(time.time() - start)
            print(f"✅ {name} ready ({elapsed}s, status {e.code})")
            return
        except (urllib.error.URLError, ConnectionError, TimeoutError) as e:
            last_err = e
        time.sleep(5)
    raise TimeoutError(
        f"{name} did not become ready in {timeout}s. Last error: {last_err}.\n"
        f"Last log lines:\n" + "-" * 60 + "\n" + _tail(log_path) + "-" * 60
    )

# Launch SEQUENTIALLY — wait for each to fully initialize before starting the next.
# Parallel launch causes vllm's memory-profiling assertion to fire because
# both processes are allocating/freeing GPU memory at the same time and each
# sees the other's churn as 'unexpected' free-memory deltas.
speech_log = open("logs/vllm-speech.log", "w")
print("⏳ Starting Granite Speech vLLM (downloads weights on first run, ~4 min)...")
speech_proc = subprocess.Popen(
    [
        VENV_VLLM, "serve", "ibm-granite/granite-speech-4.1-2b",
        "--api-key", "token-abc123",
        "--max-model-len", "2048",
        "--gpu-memory-utilization", "0.35",
        "--port", "8083",
    ],
    stdout=speech_log, stderr=subprocess.STDOUT,
)
wait_for("http://127.0.0.1:8083/v1/models", "Granite Speech (STT)", speech_proc, "logs/vllm-speech.log", timeout=1200)

switch_log = open("logs/vllm-switch.log", "w")
print("⏳ Starting Granite Switch vLLM (downloads weights on first run, ~4 min)...")
switch_proc = subprocess.Popen(
    [
        VENV_VLLM, "serve", "ibm-granite/granite-switch-4.1-3b-preview",
        "--gpu-memory-utilization", "0.35",
        # Cap context window so KV cache fits in our 0.4 GPU share. The default
        # 131072 wants ~15 GiB of KV cache; voice turns need a tiny fraction of that.
        "--max-model-len", "8192",
        "--port", "8000",
    ],
    stdout=switch_log, stderr=subprocess.STDOUT,
)
wait_for("http://127.0.0.1:8000/v1/models", "Granite Switch (LLM)", switch_proc, "logs/vllm-switch.log", timeout=1200)

print("✅ Both vLLM servers are up")

## Cell 5 — Launch backend + frontend (~30s)

- **Pipecat backend** on port 7860 (FastAPI + SmallWebRTC signaling).
- **Next.js frontend** on port 3000 (proxies WebRTC signaling to the backend in-process).

The backend reads `HF_TOKEN` and uses it to mint a TURN relay credential per session — that's how WebRTC media reaches your browser through the cloudflared tunnel.

In [ ]:
import os
import subprocess
import time
import urllib.request
import urllib.error

VENV_PY = os.path.abspath(".venv/bin/python")

# Build the frontend in production mode. Dev mode (`npm run dev`) tries to
# open a webpack-hmr WebSocket back through the cloudflared tunnel, which
# tunnels poorly and triggers dynamic-import failures that leave the chat
# UI blank. Prod mode is a static-served bundle — no HMR, no SSR weirdness.
print("⏳ Building frontend (prod mode, ~30-60s)...")
subprocess.run(
    "cd frontend && rm -rf .next && npm run build 2>&1 | tail -10",
    shell=True, check=True,
)

backend_env = {**os.environ}
backend_env.setdefault("HOST", "127.0.0.1")
backend_env.setdefault("PORT", "7860")

backend_log = open("logs/backend.log", "w")
backend_proc = subprocess.Popen(
    [VENV_PY, "-m", "granite_speech_demo.server"],
    env=backend_env,
    stdout=backend_log, stderr=subprocess.STDOUT,
)

frontend_env = {**os.environ, "PIPECAT_BACKEND_URL": "http://127.0.0.1:7860"}
frontend_log = open("logs/frontend.log", "w")
frontend_proc = subprocess.Popen(
    ["npm", "run", "start"],
    cwd="frontend",
    env=frontend_env,
    stdout=frontend_log, stderr=subprocess.STDOUT,
)

wait_for("http://127.0.0.1:7860/api/ivr/config", "Pipecat backend", backend_proc, "logs/backend.log", timeout=120)
wait_for("http://127.0.0.1:3000", "Next.js frontend", frontend_proc, "logs/frontend.log", timeout=120)
print("✅ Backend + frontend are up")

## Cell 6 — Open the public URL and talk

Starts a Cloudflare Quick Tunnel to expose `localhost:3000` on a public `*.trycloudflare.com` URL. The tunnel handles WebRTC *signaling* (HTTP/WebSocket); the *media* path goes through the TURN relay minted by the backend, so audio works even though the Colab runtime has no public IP.

**One tunnel is enough** — the frontend talks to the backend in-process via Next.js API routes.

In [ ]:
import re
import subprocess
import time

tunnel_log_path = "logs/cloudflared.log"
tunnel_log = open(tunnel_log_path, "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:3000", "--no-autoupdate"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)

url_re = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
public_url = None
deadline = time.time() + 60
while time.time() < deadline and public_url is None:
    time.sleep(2)
    with open(tunnel_log_path) as f:
        m = url_re.search(f.read())
    if m:
        public_url = m.group(0)

if not public_url:
    raise RuntimeError("cloudflared did not print a public URL. See logs/cloudflared.log")

banner = "\n".join([
    "",
    "╔" + "═" * 70 + "╗",
    "║" + "  GRANITE SPEECH DEMO IS LIVE".ljust(70) + "║",
    "╠" + "═" * 70 + "╣",
    "║" + f"  {public_url}".ljust(70) + "║",
    "║" + "".ljust(70) + "║",
    "║" + "  1. Open the URL above in Chrome / Edge / Firefox".ljust(70) + "║",
    "║" + "  2. Allow microphone access when prompted".ljust(70) + "║",
    "║" + "  3. Click the mic button and start talking".ljust(70) + "║",
    "╚" + "═" * 70 + "╝",
    "",
])
print(banner)

## If something goes wrong

Each background process writes to a file in `logs/`:

- `logs/vllm-speech.log` — Granite Speech STT server
- `logs/vllm-switch.log` — Granite Switch LLM server
- `logs/backend.log` — Pipecat backend (look here for TURN minting messages)
- `logs/frontend.log` — Next.js dev server
- `logs/cloudflared.log` — Cloudflare tunnel (the public URL is in here)

View one with `!tail -100 logs/vllm-speech.log` (or open the file from the Colab file browser).

**Common failures:**
- *T4 OOM:* switch the runtime to A100 or L4. Both Granite models won't fit on a T4.
- *`HF_TOKEN` missing:* re-run Cell 3 after adding the secret. Without it, the backend falls back to STUN-only and audio likely won't connect through the cloudflared tunnel.
- *Stuck "waiting for vLLM":* model weights are downloading. The cell waits up to 20 min — let it run.
- *Re-running cells without cleaning up:* old processes still hold the ports. Run the kill-switch cell below, then re-run from the top.

## Caveats

- The `*.trycloudflare.com` URL is public for as long as this notebook runs. Anyone with the link can join the session.
- Colab kernels die after ~24h or when idle. Restart the notebook to get a fresh URL.
- One Colab session serves one user. Each reader runs their own copy of this notebook.

## Kill switch — clean up before re-running

Run this if you need to re-run any of the launch cells. It stops the tunnel, frontend, backend, and both vLLM processes.

In [ ]:
import subprocess

# Stop tracked Popen handles from this kernel session.
for name, p in [
    ("cloudflared", globals().get("tunnel_proc")),
    ("frontend", globals().get("frontend_proc")),
    ("backend", globals().get("backend_proc")),
    ("vllm-switch", globals().get("switch_proc")),
    ("vllm-speech", globals().get("speech_proc")),
]:
    if p is not None and p.poll() is None:
        p.terminate()
        try:
            p.wait(timeout=10)
        except Exception:
            p.kill()
        print(f"🛑 stopped {name}")
    else:
        print(f"   {name}: not tracked / already dead")

# Also kill by name — catches processes whose Popen handles got lost across
# cell re-runs or kernel restarts. Without this, GPU memory stays held by
# zombie vllm processes and the next Cell 4 run fails with OOM at startup.
for pattern in ["vllm", "cloudflared tunnel", "granite_speech_demo.server", "next dev"]:
    subprocess.run(f"pkill -9 -f '{pattern}' || true", shell=True)
    print(f"🧹 pkill -9 -f '{pattern}'")
print("\nIf any vllm processes were running, GPU memory should now be freed.")
print("Run `!nvidia-smi` to confirm before re-running Cell 4.")